<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Momentum Trading: Moving Averages, Brute Force and Pyomo

A side notebook, separate from Studio 5. Ms. Womack decides **how much** of each stock to hold. Momentum trading
decides **when** to hold it: ride the trend, step aside when it breaks.

The classic rule: **hold a stock while its short moving average (say 50 days) is above its long moving average (say
200 days); otherwise sit in cash.** Traders call the moment the short average crosses above the long one a "golden
cross", and the cross below a "death cross".

* **Part 1 · Your stocks and daily prices** (the same themes as Studio 5; Connecticut to start).
* **Part 2 · The 50/200 rule on one stock**, drawn on its price chart.
* **Part 3 · Brute force: try every pair of windows.** Which pair would have made the most from 2021 to 2025?
* **Part 4 · Pyomo picks one pair per stock**, under rules (a trading budget, a linking constraint).
* **Part 5 · The honest test: 2026**, which the search never saw.

Why both brute force *and* Pyomo? Profit as a function of the two window lengths is a jagged staircase: move a window
by one day and a trade flips. There is no formula for a solver to climb, so the **search** is brute force (Module 3).
What Pyomo is great at is the **choosing**: once every pair has a score, pick one per stock under business rules.
That part is project selection (Module 5) all over again.

## Import modules

Run this first. It installs Pyomo with the **cbc** solver (for yes/no choices with a linear objective) and yfinance.

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre yfinance
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd
import time
from pyomo.environ import *
from IPython.display import display

import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)

cbc_path = '/content/bin/cbc'

# Part 1 · Your stocks and daily prices

#### 🎯 Your choice

Same themes as Studio 5 (see the infographic there). Connecticut companies to start: Travelers, The Hartford,
Cigna, Stanley Black & Decker, RTX, Otis and friends.

In [ ]:
# 'starter', or a theme: 'connecticut', 'ai_chips', 'oil_and_gas', 'banks_and_payments', 'dividends', 'healthcare',
# 'food_and_restaurants', 'travel', 'defense_and_aerospace', 'utilities_and_power', 'shopping', 'sports_and_entertainment'
my_theme = 'connecticut'

In [ ]:
themes = {
    'starter': ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'META', 'JPM', 'GS', 'V', 'JNJ', 'LLY', 'UNH', 'XOM', 'CVX', 'KO', 'WMT', 'COST', 'AMZN', 'HD', 'CAT', 'LMT'],
    'connecticut': ['TRV', 'HIG', 'CI', 'SWK', 'RTX', 'OTIS', 'CARR', 'URI', 'SYF', 'HUBB', 'ES', 'IT', 'FDS', 'CHTR', 'XOM', 'PG', 'MSFT', 'UNH'],
    'ai_chips': ['NVDA', 'AMD', 'AVGO', 'MU', 'LRCX', 'AMAT', 'KLAC', 'INTC', 'QCOM', 'TXN', 'ADI', 'ANET', 'MSFT', 'GOOGL', 'META', 'ORCL', 'LLY', 'XOM'],
    'oil_and_gas': ['XOM', 'CVX', 'COP', 'EOG', 'SLB', 'OXY', 'PSX', 'MPC', 'VLO', 'KMI', 'WMB', 'OKE', 'HAL', 'DVN', 'FANG', 'MSFT', 'JNJ'],
    'banks_and_payments': ['JPM', 'BAC', 'WFC', 'C', 'GS', 'MS', 'SCHW', 'V', 'MA', 'AXP', 'PYPL', 'COF', 'BLK', 'USB', 'AAPL', 'UNH', 'CVX'],
    'dividends': ['KO', 'PEP', 'PG', 'JNJ', 'MRK', 'ABBV', 'VZ', 'T', 'DUK', 'SO', 'XOM', 'CVX', 'MMM', 'IBM', 'MO', 'PM', 'CSCO', 'O'],
    'healthcare': ['LLY', 'JNJ', 'UNH', 'ABBV', 'MRK', 'PFE', 'TMO', 'ABT', 'ISRG', 'AMGN', 'GILD', 'CVS', 'MDT', 'BSX', 'SYK', 'AAPL', 'XOM'],
    'food_and_restaurants': ['MCD', 'SBUX', 'CMG', 'YUM', 'DPZ', 'KO', 'PEP', 'MDLZ', 'HSY', 'GIS', 'KHC', 'KDP', 'DRI', 'TSN', 'COST', 'AAPL', 'JNJ', 'XOM'],
    'travel': ['BKNG', 'MAR', 'HLT', 'ABNB', 'EXPE', 'DAL', 'UAL', 'LUV', 'RCL', 'CCL', 'NCLH', 'LVS', 'WYNN', 'MGM', 'UBER', 'AAPL', 'JNJ', 'XOM'],
    'defense_and_aerospace': ['LMT', 'RTX', 'NOC', 'GD', 'BA', 'LHX', 'HII', 'TXT', 'GE', 'HON', 'TDG', 'HWM', 'PLTR', 'LDOS', 'XOM', 'JNJ', 'MSFT'],
    'utilities_and_power': ['NEE', 'DUK', 'SO', 'D', 'AEP', 'EXC', 'SRE', 'XEL', 'PEG', 'ED', 'CEG', 'VST', 'ETR', 'FSLR', 'AES', 'XOM', 'JNJ', 'MSFT'],
    'shopping': ['AMZN', 'WMT', 'COST', 'TGT', 'HD', 'LOW', 'TJX', 'NKE', 'ROST', 'ULTA', 'DG', 'DLTR', 'BBY', 'EBAY', 'AAPL', 'CVS', 'XOM'],
    'sports_and_entertainment': ['DIS', 'NFLX', 'LYV', 'NKE', 'TTWO', 'CMCSA', 'CHTR', 'MAR', 'RCL', 'LVS', 'FOXA', 'HAS', 'GOOGL', 'META', 'AAPL', 'JNJ', 'XOM'],
}
if my_theme in themes:
    stocks = themes[my_theme]
else:
    print("Unknown theme:", my_theme, "- using connecticut")
    stocks = themes['connecticut']
print(len(stocks), "stocks:", ", ".join(stocks))

### Daily prices, back to 2020

Momentum needs **daily** prices (Studio 5 used monthly), and a 200-day average needs 200 days of history before the
first trade, so we start in January 2020. One download for all of your stocks plus SPY; if Yahoo blocks Colab, the
class copy is used instead.

In [ ]:
import yfinance as yf

settings = {
    'download_from': '2020-01-01',
    'fit_start': '2021-01-01',    # the search learns from 2021 to 2025...
    'fit_end': '2025-12-31',
    'test_start': '2026-01-01',   # ...and is judged on 2026, which it never sees
    'trading_cost': 0.001,        # 0.1% of the money every time a stock goes in or out of the market
}

prices = None
try:
    data = yf.download(stocks + ['SPY'], start=settings['download_from'], interval='1d', auto_adjust=True, progress=False)
    prices = data['Close']
except Exception as problem:
    print("Yahoo did not answer:", problem)
if prices is None or 'SPY' not in prices.columns or prices.shape[1] < 0.8 * (len(stocks) + 1):
    print("Using the class copy of daily prices instead.")
    saved = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/momentum/data/daily_prices_2020_2026.csv', index_col=0, parse_dates=True)
    keep = []
    for t in stocks + ['SPY']:
        if t in saved.columns:
            keep.append(t)
    prices = saved[keep]

# today's row is not a close until the market has closed, so we stop at yesterday
prices = prices[prices.index < pd.Timestamp.today().normalize()]
usable = []
for t in stocks:
    if t in prices.columns and prices[t].notna().sum() > 1000:
        usable.append(t)
stocks = usable
print(prices.shape[0], "trading days, from", prices.index[0].date(), "to", prices.index[-1].date(), "|", len(stocks), "stocks")

# Part 2 · The 50/200 rule on one stock

**The rule, in four lines:**

1. The **short average**: the average closing price of the last 50 trading days (about 2½ months).
2. The **long average**: the same over the last 200 trading days (about 10 months).
3. If the short average is **above** the long one at today's close, hold the stock **tomorrow**; otherwise hold cash.
4. Every switch in or out costs 0.1% of the money (a stand-in for fees and spreads).

"Tomorrow" matters: the rule only uses prices it could have seen. `run_rule` below does all four lines for one
stock, between two dates.

In [ ]:
def run_rule(price, short_days, long_days, start, end, cost):
    # one stock, one pair of windows: $10,000, in the market only while short average > long average
    short_avg = price.rolling(short_days).mean()
    long_avg = price.rolling(long_days).mean()
    # decide at today's close, hold tomorrow (no peeking)
    in_market = (short_avg > long_avg).astype(float).shift(1)
    daily = price.pct_change()

    window = (price.index >= start) & (price.index <= end)
    in_market = in_market[window].fillna(0)
    daily = daily[window].fillna(0)
    switches = in_market.diff().abs()
    switches.iloc[0] = in_market.iloc[0]

    strategy = in_market * daily - cost * switches
    path = 10000 * (1 + strategy).cumprod()
    return {'path': path, 'in_market': in_market, 'trades': int(switches.sum()),
            'short_avg': short_avg[window], 'long_avg': long_avg[window]}


def buy_and_hold(price, start, end):
    window = price[(price.index >= start) & (price.index <= end)]
    return 10000 * window / window.iloc[0]


print("run_rule is ready")

Pick a stock and two windows. Change `showcase` to any of your stocks.

In [ ]:
showcase = stocks[0]
short_days = 50
long_days = 200

one = run_rule(prices[showcase], short_days, long_days, settings['fit_start'], settings['fit_end'], settings['trading_cost'])
hold_path = buy_and_hold(prices[showcase], settings['fit_start'], settings['fit_end'])
price = prices[showcase][one['path'].index]

fig, (top_panel, bottom_panel) = subplots(2, 1, figsize=(13, 8), sharex=True)
top_panel.plot(price.index, price.values, color='black', linewidth=1, label=showcase + ' price')
top_panel.plot(one['short_avg'].index, one['short_avg'].values, color='tab:orange', label='%d-day average' % short_days)
top_panel.plot(one['long_avg'].index, one['long_avg'].values, color='tab:blue', label='%d-day average' % long_days)
top_panel.fill_between(price.index, price.min(), price.max(), where=one['in_market'] > 0, color='tab:green', alpha=0.12,
                       label='in the market')
went_in = one['in_market'].diff() > 0
went_out = one['in_market'].diff() < 0
top_panel.scatter(price.index[went_in], price[went_in], marker='^', color='tab:green', s=70, zorder=5, label='buy')
top_panel.scatter(price.index[went_out], price[went_out], marker='v', color='tab:red', s=70, zorder=5, label='sell')
top_panel.set_title('%s with the %d/%d rule, 2021 to 2025' % (showcase, short_days, long_days))
top_panel.legend(fontsize=8, loc='upper left')

bottom_panel.plot(one['path'].index, one['path'].values, color='tab:orange', linewidth=2, label='the %d/%d rule' % (short_days, long_days))
bottom_panel.plot(hold_path.index, hold_path.values, color='gray', linewidth=2, label='buy and hold')
bottom_panel.axhline(10000, color='lightgray', linestyle='--')
bottom_panel.set_ylabel('value of $10,000')
bottom_panel.legend(fontsize=9)
show()
print("the rule: $%.0f after %d trades | buy and hold: $%.0f" % (one['path'].iloc[-1], one['trades'], hold_path.iloc[-1]))

**Read the chart.** Green shading is when the rule holds the stock. The rule never buys the bottom or sells the
top: averages are slow, so it gets in after a rise has started and out after a fall has started. It earns its keep
by **sitting out long slides**. Did it beat buy and hold for your stock?

# Part 3 · Brute force: try every pair of windows

Is 50/200 special? Try every pair in the grid below on 2021 to 2025 and see. This is the brute-force search from
Module 3: no cleverness, just try them all. Each backtest takes a few thousandths of a second, so every pair for
every stock takes only seconds.

In [ ]:
grid = {
    'short_days': [10, 20, 30, 40, 50, 60, 80, 100],
    'long_days': [50, 75, 100, 125, 150, 175, 200, 250],
}

rows = []
t0 = time.time()
for s in stocks:
    hold_end = buy_and_hold(prices[s], settings['fit_start'], settings['fit_end']).iloc[-1]
    rows.append({'stock': s, 'choice': 'hold', 'short': 0, 'long': 0, 'fit_value': hold_end, 'trades': 1})
    for short_w in grid['short_days']:
        for long_w in grid['long_days']:
            if short_w >= long_w:
                continue
            result = run_rule(prices[s], short_w, long_w, settings['fit_start'], settings['fit_end'], settings['trading_cost'])
            rows.append({'stock': s, 'choice': '%d/%d' % (short_w, long_w), 'short': short_w, 'long': long_w,
                         'fit_value': result['path'].iloc[-1], 'trades': result['trades']})
search = pd.DataFrame(rows)
print("%d backtests in %.1f seconds" % (len(search), time.time() - t0))

### The heat map for your showcase stock

Each square is one pair of windows: what \$10,000 became from 2021 to 2025. The star is the best pair; the circle is
the textbook 50/200. **Look at the shape, not just the star.** A lone bright square is probably luck. A broad bright
region means the idea works across many settings, which you can trust more.

In [ ]:
mine = search[(search['stock'] == showcase) & (search['choice'] != 'hold')]
table = mine.pivot(index='short', columns='long', values='fit_value')
best = mine.loc[mine['fit_value'].idxmax()]
hold_value = search[(search['stock'] == showcase) & (search['choice'] == 'hold')]['fit_value'].iloc[0]

figure(figsize=(10, 6))
imshow(table.values, cmap='RdYlGn', aspect='auto', origin='lower')
colorbar(label='$10,000 became (2021 to 2025)')
xticks(range(len(table.columns)), table.columns)
yticks(range(len(table.index)), table.index)
for i in range(len(table.index)):
    for j in range(len(table.columns)):
        if not np.isnan(table.values[i, j]):
            text(j, i, '%.0f' % (table.values[i, j] / 1000) + 'k', ha='center', va='center', fontsize=8)
best_i = list(table.index).index(best['short'])
best_j = list(table.columns).index(best['long'])
plot(best_j, best_i, marker='*', markersize=22, color='black', linestyle='none', label='best pair: %s' % best['choice'])
if 50 in list(table.index) and 200 in list(table.columns):
    plot(list(table.columns).index(200), list(table.index).index(50), marker='o', markersize=18, markerfacecolor='none',
         markeredgecolor='black', linestyle='none', label='the textbook 50/200')
xlabel('long window (days)')
ylabel('short window (days)')
title('%s: every pair of windows, 2021 to 2025 (buy and hold: $%.0f)' % (showcase, hold_value))
legend(loc='upper left', fontsize=8)
show()

### The best pair for every stock

The best pair in hindsight for each of your stocks, next to simply buying and holding it.

In [ ]:
best_rows = []
for s in stocks:
    one_stock = search[search['stock'] == s]
    top = one_stock.loc[one_stock['fit_value'].idxmax()]
    hold_end = one_stock[one_stock['choice'] == 'hold']['fit_value'].iloc[0]
    best_rows.append({'stock': s, 'best choice': top['choice'], 'best $10,000 became': round(top['fit_value']),
                      'buy and hold': round(hold_end), 'trades': top['trades']})
best_table = pd.DataFrame(best_rows).set_index('stock')
best_table.sort_values('best $10,000 became', ascending=False)

# Part 4 · Pyomo picks: one pair per stock, under rules

Picking the best pair per stock is easy without Pyomo (take the top row). It gets interesting with **rules that tie
the stocks together**, and that is a yes/no model, like project selection:

$$z_{s,c} = \begin{cases} 1 & \text{stock } s \text{ uses choice } c \text{ (a pair of windows, or buy and hold)} \\ 0 & \text{otherwise} \end{cases}$$

| Rule | Constraint |
|---|---|
| each stock gets exactly one choice | $\sum_c z_{s,c} = 1$ for every stock |
| the long window at least 3× the short one | $z_{s,c} = 0$ for pairs that break it |
| a trading budget for the whole fund | $\sum_{s,c} \text{trades}_{s,c}\, z_{s,c} \le$ budget |
| at most $M$ stocks trade on momentum | $\sum_s \sum_{c \ne \text{hold}} z_{s,c} \le M$ |
| *optional:* one pair of windows for everyone | pick $w_c$ (one pair), and $z_{s,c} \le w_c$, $\sum_c w_c = 1$ |

The objective: the most money at the end of 2025, with the same \$10,000 in each stock (so the fund is the average of
its stocks).

In [ ]:
pick_rules = {
    'long_at_least_times_short': 3,     # the long window must be at least 3 times the short one
    'trades_per_stock_per_year': 4,     # the fund's trading budget: on average, at most this many trades per stock a year
    'max_momentum_stocks': None,        # e.g. 8: at most 8 stocks trade on momentum, the rest buy and hold
    'one_pair_for_everyone': False,     # True: every momentum stock must use the same two windows (a linking rule)
}


def pick_choices(rules_now):
    years = (pd.Timestamp(settings['fit_end']) - pd.Timestamp(settings['fit_start'])).days / 365.25
    keys = []
    for i in range(len(search)):
        keys.append(search['stock'].iloc[i] + '|' + search['choice'].iloc[i])
    score = dict(zip(keys, search['fit_value']))
    trades = dict(zip(keys, search['trades']))
    pairs = []
    for c in search['choice'].unique():
        if c != 'hold':
            pairs.append(c)

    model = ConcreteModel()
    model.z = Var(keys, domain=Binary)
    objective_expr = 0
    for k in keys:
        objective_expr += score[k] * model.z[k] / len(stocks)
    model.fund_value = Objective(expr=objective_expr, sense=maximize)
    model.constraints = ConstraintList()

    for s in stocks:
        one_choice = 0
        for k in keys:
            if k.split('|')[0] == s:
                one_choice += model.z[k]
        model.constraints.add(one_choice == 1)

    for i in range(len(search)):
        row = search.iloc[i]
        if row['choice'] != 'hold' and row['long'] < rules_now['long_at_least_times_short'] * row['short']:
            model.constraints.add(model.z[keys[i]] == 0)

    budget = 0
    for k in keys:
        budget += trades[k] * model.z[k]
    model.constraints.add(budget <= rules_now['trades_per_stock_per_year'] * years * len(stocks))

    if rules_now['max_momentum_stocks'] is not None:
        momentum = 0
        for k in keys:
            if not k.endswith('|hold'):
                momentum += model.z[k]
        model.constraints.add(momentum <= rules_now['max_momentum_stocks'])

    if rules_now['one_pair_for_everyone']:
        model.w = Var(pairs, domain=Binary)
        chosen_pairs = 0
        for c in pairs:
            chosen_pairs += model.w[c]
        model.constraints.add(chosen_pairs == 1)
        for k in keys:
            c = k.split('|')[1]
            if c != 'hold':
                model.constraints.add(model.z[k] <= model.w[c])

    results = SolverFactory('cbc', executable=cbc_path).solve(model)
    if results.solver.termination_condition != TerminationCondition.optimal:
        return None
    picks = {}
    for k in keys:
        if model.z[k]() > 0.5:
            picks[k.split('|')[0]] = k.split('|')[1]
    return {'picks': picks, 'fund_value': model.fund_value()}


chosen = pick_choices(pick_rules)
if chosen is None:
    print("The rules cannot all be met: loosen the trading budget or the window rule.")
else:
    pick_table = pd.DataFrame({'Pyomo picked': pd.Series(chosen['picks'])})
    pick_table['best in hindsight'] = best_table['best choice']
    print("the fund on 2021 to 2025 with Pyomo's picks: $%.0f" % chosen['fund_value'])
    display(pick_table)

**Play with the rules.** Tighten the budget (`'trades_per_stock_per_year': 2`), cap the momentum stocks
(`'max_momentum_stocks': 6`), or force one pair for everyone (`'one_pair_for_everyone': True`) and run the cell again.
The next cell compares a few rule sets side by side: the price of each rule, like Studio 5's ladder.

In [ ]:
rule_sets = {
    'no rules (best in hindsight)': {'long_at_least_times_short': 0, 'trades_per_stock_per_year': 1000, 'max_momentum_stocks': None, 'one_pair_for_everyone': False},
    'your pick_rules': pick_rules,
    'one pair for everyone': {'long_at_least_times_short': pick_rules['long_at_least_times_short'], 'trades_per_stock_per_year': pick_rules['trades_per_stock_per_year'], 'max_momentum_stocks': pick_rules['max_momentum_stocks'], 'one_pair_for_everyone': True},
    'tight budget: 2 trades a year': {'long_at_least_times_short': pick_rules['long_at_least_times_short'], 'trades_per_stock_per_year': 2, 'max_momentum_stocks': pick_rules['max_momentum_stocks'], 'one_pair_for_everyone': False},
}
compare = {}
picks_by_rules = {}
for name in rule_sets:
    answer = pick_choices(rule_sets[name])
    if answer is not None:
        compare[name] = round(answer['fund_value'])
        picks_by_rules[name] = answer['picks']
pd.Series(compare, name='fund value at the end of 2025 ($)').to_frame()

# Part 5 · The honest test: 2026

Everything above was judged on 2021 to 2025, the same years the search looked at. Picking the best of many pairs on
the past always looks good on the past. Now hold each choice through **2026**, which the search never saw, and
compare with simply buying and holding the same stocks, the textbook 50/200 for every stock, and the S&P 500.

In [ ]:
end_day = str(prices.index[-1].date())


def fund_in_2026(picks):
    # the same $10,000 in each stock, each stock following its own choice; the fund is the sum
    total = None
    for s in stocks:
        c = picks[s]
        if c == 'hold':
            path = buy_and_hold(prices[s], settings['test_start'], end_day)
        else:
            windows = c.split('/')
            path = run_rule(prices[s], int(windows[0]), int(windows[1]), settings['test_start'], end_day,
                            settings['trading_cost'])['path']
        if total is None:
            total = path / len(stocks)
        else:
            total = total + path / len(stocks)
    return total


textbook = {}
holding = {}
for s in stocks:
    textbook[s] = '50/200'
    holding[s] = 'hold'
race = {}
for name in picks_by_rules:
    race['Pyomo: ' + name] = fund_in_2026(picks_by_rules[name])
race['the textbook 50/200 for every stock'] = fund_in_2026(textbook)
race['buy and hold, even split'] = fund_in_2026(holding)
race['S&P 500 (SPY)'] = buy_and_hold(prices['SPY'], settings['test_start'], end_day)

figure(figsize=(13, 5.5))
for name in race:
    width = 1.5
    color = None
    if name == 'S&P 500 (SPY)':
        width = 3
        color = 'black'
    if name == 'buy and hold, even split':
        color = 'gray'
    plot(race[name].index, race[name].values, linewidth=width, color=color, label=name)
axhline(10000, color='lightgray', linestyle='--')
ylabel('value of $10,000')
title('2026, the year the search never saw')
legend(fontsize=8)
show()

scores = {}
for name in race:
    scores[name] = round(race[name].iloc[-1])
pd.Series(scores, name='$10,000 became in 2026').sort_values(ascending=False).to_frame()

### Did the best pair stay the best? (overfitting, in one picture)

Every pair of windows for your showcase stock: across, what it made on 2021 to 2025; up, what it made in 2026. If the
past predicted the future, the dots would climb from left to right in a tidy line. (A flat row of dots means many
pairs did exactly the same thing in 2026: they all stayed in the market the whole time.)

In [ ]:
fit_values = []
test_values = []
for i in range(len(mine)):
    row = mine.iloc[i]
    later = run_rule(prices[showcase], int(row['short']), int(row['long']), settings['test_start'], end_day, settings['trading_cost'])
    fit_values.append(row['fit_value'])
    test_values.append(later['path'].iloc[-1])

figure(figsize=(8, 6))
scatter(fit_values, test_values, color='tab:blue', alpha=0.7)
best_position = list(mine['choice']).index(best['choice'])
scatter([fit_values[best_position]], [test_values[best_position]], marker='*', s=300, color='black', label='best pair on 2021-2025')
axhline(10000, color='lightgray', linestyle='--')
xlabel('$10,000 became, 2021 to 2025 (what the search saw)')
ylabel('$10,000 became, 2026 (what it did not see)')
title('%s: does the past ranking of window pairs hold up?' % showcase)
legend()
show()
print("correlation between the two: %.2f (1 = the past ranking held perfectly, 0 = no relation)" % np.corrcoef(fit_values, test_values)[0, 1])

### Your turn

1. For your showcase stock, did the 50/200 rule beat buy and hold on 2021 to 2025? Where on the chart did it win or
   lose?
2. Is your heat map one bright square or a broad bright region? What does that tell you about trusting the best pair?
3. What did the trading budget and "one pair for everyone" cost on 2021 to 2025? Did either help in 2026?
4. Brute force found the lags and Pyomo picked among them. Why could Pyomo not search the lags directly?
5. Momentum or Ms. Womack: which would you trust with your own money for the rest of 2026, and why?